<a href="https://colab.research.google.com/github/AntonyJohny/Innomatics_Research_Labs/blob/main/IN126001202_GenAI/Task_3/Task_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Using DialoGPT-medium for a balance of speed and intelligence
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model loaded on {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded on cpu


In [13]:
knowledge_base = {
    "what is artificial intelligence": "Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.",
    "who created python": "Python was created by Guido van Rossum and released in 1991."
}
def start_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    chat_history_ids = None

    while True:
        user_input = input("User: ").strip()

        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye!")
            break

        # 1. Check Knowledge Base first
        if user_input.lower().strip('?') in knowledge_base:
            response = knowledge_base[user_input.lower().strip('?')]
            print(f"Chatbot: {response}")
            continue # Skip the model generation for this turn

        # 2. Otherwise, use the Transformer Model for general chat
        new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt').to(device)
        bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1) if chat_history_ids is not None else new_user_input_ids

        chat_history_ids = model.generate(
            bot_input_ids,
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,
            do_sample=True,
            top_k=50,
            temperature=0.7,
            attention_mask=torch.ones(bot_input_ids.shape, device=device)
        )

        response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
        print(f"Chatbot: {response}")

start_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?
User: Hello
Chatbot: Hello : 3
User: What is Artificial Intelligence?
Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.
User: Who created Python?
Chatbot: Python was created by Guido van Rossum and released in 1991.
User: Thank you
Chatbot: You're welcome
User: exit
Chatbot: Goodbye!
